# **Phase 5**

In [39]:
import pandas as pd
import numpy as np
import json
import sqlite3
import os, glob
from pathlib import Path
from datetime import datetime, timezone, timedelta
import warnings
warnings.filterwarnings("ignore")

import xgboost as xgb

print("✓ Libraries loaded")


✓ Libraries loaded


In [40]:
def find_file(preferred_path, filename):
    if os.path.exists(preferred_path):
        return preferred_path
    candidates = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if candidates:
        print(f"⚠ Auto-detected {filename} at: {candidates[0]}")
        return candidates[0]
    raise FileNotFoundError(
        f"{filename} not found. Run Phases 1-4 first, or upload their outputs as a dataset."
    )

WORKDIR = "."

priority = pd.read_csv(find_file(f"{WORKDIR}/enforcement_ranking_with_cost.csv", "enforcement_ranking_with_cost.csv"))
ts       = pd.read_csv(find_file(f"{WORKDIR}/daily_timeseries_features.csv", "daily_timeseries_features.csv"))
df       = pd.read_csv(find_file(f"{WORKDIR}/violations_geocoded.csv", "violations_geocoded.csv"))
ts["date"] = pd.to_datetime(ts["date"])

print(f"✓ Priority/cost table : {len(priority):,} ranked zones")
print(f"✓ Time series         : {len(ts):,} rows")
print(f"✓ Raw violations      : {len(df):,} rows  (used for device→station mapping)")
print(f"✓ Forecast model loaded from {model_path}")


✓ Priority/cost table : 843 ranked zones
✓ Time series         : 14,403 rows
✓ Raw violations      : 122,404 rows  (used for device→station mapping)
✓ Forecast model loaded from cat_model.pkl


In [41]:
priority.columns

Index(['priority_rank', 'junction_name', 'n_violations', 'total_risk', 'cii',
       'cii_tier', 'dominant_type', 'dominant_vehicle', 'dominant_station',
       'lat', 'lon', 'h3_cell', 'total_overlap_risk', 'avg_active_violations',
       'max_active_violations', 'avg_overlap_multiplier',
       'max_overlap_multiplier', 'road_factor', 'congestion_weight',
       'vehicle_weight', 'delay_score', 'congestion_intensity', 'delay_index',
       'vehicle_minutes_lost', 'person_minutes_lost', 'person_hours_lost',
       'congestion_cost_rs', 'traffic_severity', 'traffic_impact_score',
       'traffic_rank'],
      dtype='object')

In [42]:
ts.head()

,h3_cell,date,total_risk,total_chalans,avg_neighbor_risk,avg_neighbor_risk_lag_1,avg_neighbor_risk_lag_2,avg_neighbor_risk_lag_3,avg_neighbor_risk_lag_7,nbr_roll_mean_7,nbr_roll_std_7,total_risk_lag1,total_risk_lag2,total_risk_lag3,total_risk_lag7,dow
0,8860144a2dfffff,2023-11-29,48,4,30.0,-999.0,-999.0,-999.0,-999.0,NaN,NaN,-999.0,-999.0,-999.0,-999.0,2
1,8860144b4dfffff,2023-11-15,57,19,NaN,-999.0,-999.0,-999.0,-999.0,NaN,NaN,-999.0,-999.0,-999.0,-999.0,2
2,8860144b5bfffff,2023-11-29,30,3,48.0,-999.0,-999.0,-999.0,-999.0,NaN,NaN,-999.0,-999.0,-999.0,-999.0,2
3,8860144b5bfffff,2023-11-30,12,1,NaN,48.0,-999.0,-999.0,-999.0,48.0,NaN,30.0,-999.0,-999.0,-999.0,3
4,8860144b5bfffff,2023-12-01,15,1,NaN,-999.0,48.0,-999.0,-999.0,48.0,NaN,12.0,30.0,-999.0,-999.0,4


In [43]:
import joblib

model = joblib.load("cat_model.pkl")

print(type(model))

<class 'catboost.core.CatBoostRegressor'>


In [44]:
# ============================================================
# FORECAST CONGESTION RISK FOR EACH H3 CELL
# ============================================================

FEATURES = [
    'avg_neighbor_risk_lag_1',
    'avg_neighbor_risk_lag_2',
    'avg_neighbor_risk_lag_3',
    'avg_neighbor_risk_lag_7',
    'total_risk_lag1',
    'total_risk_lag2',
    'total_risk_lag3',
    'total_risk_lag7',
    'nbr_roll_std_7',
    'dow'
]

# Most recent observation for every H3 cell
latest = (
    ts.sort_values("date")
      .groupby("h3_cell")
      .tail(1)
      .copy()
)

# Predict congestion risk
latest["predicted_risk"] = np.clip(
    model.predict(latest[FEATURES]),
    0,
    None
).round(1)

# Recent baseline risk (last 3 days)
latest["recent_mean_risk"] = latest[
    ["total_risk_lag1",
     "total_risk_lag2",
     "total_risk_lag3"]
].mean(axis=1)

# Percentage change vs recent history
latest["risk_change_pct"] = (
    (latest["predicted_risk"] - latest["recent_mean_risk"])
    / latest["recent_mean_risk"].replace(0, np.nan)
    * 100
)

# Trend classification
latest["trend"] = pd.cut(
    latest["risk_change_pct"],
    bins=[-np.inf, -10, 10, np.inf],
    labels=["FALLING", "STABLE", "RISING"]
)

# ============================================================
# MERGE FORECASTS INTO PRIORITY TABLE
# ============================================================

merged = priority.merge(
    latest[
        [
            "h3_cell",
            "predicted_risk",
            "risk_change_pct",
            "trend"
        ]
    ],
    on="h3_cell",
    how="left"
)

# Handle H3 cells with no forecast
merged["predicted_risk"] = merged["predicted_risk"].fillna(0)
merged["risk_change_pct"] = merged["risk_change_pct"].fillna(0)

merged["trend"] = (
    merged["trend"]
    .astype("object")
    .fillna("NO DATA")
)

print(f"✓ Live forecasts merged into {len(merged)} priority zones")

merged[
    [
        "junction_name",
        "cii_tier",
        "predicted_risk",
        "risk_change_pct",
        "trend"
    ]
].head(10)

✓ Live forecasts merged into 843 priority zones


,junction_name,cii_tier,predicted_risk,risk_change_pct,trend
0,BTP051 - Safina Plaza Junction,SEVERE,38.2,172.857143,RISING
1,BTP051 - Safina Plaza Junction,SEVERE,25.5,21.428571,RISING
2,BTP051 - Safina Plaza Junction,SEVERE,19.6,292.000000,RISING
3,BTP051 - Safina Plaza Junction,SEVERE,21.1,251.666667,RISING
4,BTP051 - Safina Plaza Junction,SEVERE,28.7,2.500000,STABLE
5,BTP051 - Safina Plaza Junction,SEVERE,37.2,-19.130435,FALLING
6,BTP051 - Safina Plaza Junction,SEVERE,32.1,153.421053,RISING
7,BTP051 - Safina Plaza Junction,SEVERE,23.4,56.000000,RISING
8,BTP051 - Safina Plaza Junction,SEVERE,45.7,-30.757576,FALLING
9,BTP051 - Safina Plaza Junction,SEVERE,55.5,32.142857,RISING


In [45]:
def alert_level(row):
    if row["cii_tier"] in ("SEVERE", "HIGH") and row["trend"] == "RISING":
        return "CRITICAL"
    if row["cii_tier"] in ("SEVERE", "HIGH"):
        return "WATCH"
    if row["trend"] == "RISING":
        return "WATCH"
    return "INFO"

merged["alert_level"] = merged.apply(alert_level, axis=1)

print("✓ Alert levels assigned:")
print(merged["alert_level"].value_counts())
print()
print("CRITICAL alerts:")
merged[merged["alert_level"]=="CRITICAL"][["junction_name","dominant_station","cii_tier","predicted_risk","trend"]]


✓ Alert levels assigned:
alert_level
WATCH       477
INFO        266
CRITICAL    100
Name: count, dtype: int64

CRITICAL alerts:


,junction_name,dominant_station,cii_tier,predicted_risk,trend
0,BTP051 - Safina Plaza Junction,Shivajinagar,SEVERE,38.2,RISING
1,BTP051 - Safina Plaza Junction,Shivajinagar,SEVERE,25.5,RISING
2,BTP051 - Safina Plaza Junction,Shivajinagar,SEVERE,19.6,RISING
3,BTP051 - Safina Plaza Junction,Shivajinagar,SEVERE,21.1,RISING
6,BTP051 - Safina Plaza Junction,Shivajinagar,SEVERE,32.1,RISING
...,...,...,...,...,...
282,BTP052 - CID Carlton House Junction,Cubbon Park,HIGH,32.4,RISING
283,BTP052 - CID Carlton House Junction,Cubbon Park,HIGH,35.3,RISING
284,BTP052 - CID Carlton House Junction,Cubbon Park,HIGH,96.6,RISING
285,BTP034 - LRDE Junction,High ground,HIGH,25.7,RISING


In [46]:
device_station = df.groupby("device_id")["police_station"].agg(
    lambda x: x.value_counts().index[0]
)
station_devices = device_station.reset_index().groupby("police_station")["device_id"].apply(list).to_dict()

print(f"✓ Built device roster from real data: {device_station.nunique()} stations with mapped devices")
print(f"  Example — Shivajinagar devices: {station_devices.get('Shivajinagar', [])[:5]}")


✓ Built device roster from real data: 54 stations with mapped devices
  Example — Shivajinagar devices: ['FKDEV00004', 'FKDEV00012', 'FKDEV00037', 'FKDEV00041', 'FKDEV00063']


In [47]:
def assign_units(alerts_df, station_devices):
    used = set()
    assigned = []
    for _, row in alerts_df.iterrows():
        station = row["dominant_station"]
        candidates = [d for d in station_devices.get(station, []) if d not in used]
        if candidates:
            unit = candidates[0]
            used.add(unit)
        else:
            unit = "UNASSIGNED"
        assigned.append(unit)
    return assigned

# Assign units to CRITICAL and WATCH alerts only — INFO zones don't need a dispatched unit
active_alerts = merged[merged["alert_level"].isin(["CRITICAL","WATCH"])].copy()
active_alerts = active_alerts.sort_values("alert_level", ascending=True)  # CRITICAL (C) before WATCH (W) alphabetically works here too, but be explicit:
active_alerts["alert_priority"] = active_alerts["alert_level"].map({"CRITICAL":0,"WATCH":1})
active_alerts = active_alerts.sort_values("alert_priority").reset_index(drop=True)

active_alerts["assigned_unit"] = assign_units(active_alerts, station_devices)

print(f"✓ Units assigned to {len(active_alerts)} active alerts")
n_unassigned = (active_alerts["assigned_unit"]=="UNASSIGNED").sum()
print(f"  Unassigned (station device pool exhausted): {n_unassigned}")
active_alerts[["junction_name","dominant_station","alert_level","assigned_unit"]].head(15)


✓ Units assigned to 577 active alerts
  Unassigned (station device pool exhausted): 21


,junction_name,dominant_station,alert_level,assigned_unit
0,BTP051 - Safina Plaza Junction,Shivajinagar,CRITICAL,FKDEV00004
1,BTP032 - Windsor Circle,Shivajinagar,CRITICAL,FKDEV00012
2,BTP032 - Windsor Circle,Shivajinagar,CRITICAL,FKDEV00037
3,BTP032 - Windsor Circle,Shivajinagar,CRITICAL,FKDEV00041
4,BTP032 - Windsor Circle,Shivajinagar,CRITICAL,FKDEV00063
5,"BTP016 - 5th Main Road, RPC Layout",Vijayanagara,CRITICAL,FKDEV00006
6,"BTP016 - 5th Main Road, RPC Layout",Vijayanagara,CRITICAL,FKDEV00050
7,"BTP016 - 5th Main Road, RPC Layout",Vijayanagara,CRITICAL,FKDEV00055
8,"BTP148 - 17th Main, Doopanahalli Bus Stop",Jeevanbheemanagar,CRITICAL,FKDEV00024
9,"BTP148 - 17th Main, Doopanahalli Bus Stop",Jeevanbheemanagar,CRITICAL,FKDEV00051


In [48]:
active_alerts.columns

Index(['priority_rank', 'junction_name', 'n_violations', 'total_risk', 'cii',
       'cii_tier', 'dominant_type', 'dominant_vehicle', 'dominant_station',
       'lat', 'lon', 'h3_cell', 'total_overlap_risk', 'avg_active_violations',
       'max_active_violations', 'avg_overlap_multiplier',
       'max_overlap_multiplier', 'road_factor', 'congestion_weight',
       'vehicle_weight', 'delay_score', 'congestion_intensity', 'delay_index',
       'vehicle_minutes_lost', 'person_minutes_lost', 'person_hours_lost',
       'congestion_cost_rs', 'traffic_severity', 'traffic_impact_score',
       'traffic_rank', 'predicted_risk', 'risk_change_pct', 'trend',
       'alert_level', 'alert_priority', 'assigned_unit'],
      dtype='object')

In [50]:
def build_scita_payload(row, alert_id):
    return {
        "alert_id": alert_id,
        "source_system": "AI_PARKING_INTELLIGENCE_V1",
        "h3_cell": row["h3_cell"],
        "junction_name": row["junction_name"],
        "police_station": row["dominant_station"],
        "location": {"lat": float(row["lat"]), "lon": float(row["lon"])},
        "violation_summary": {
            "dominant_type": row["dominant_type"],
            "dominant_vehicle": row["dominant_vehicle"],
            "n_violations_historical": int(row["n_violations"]),
            "total_risk_score": int(row["total_risk"]),
        },
        "congestion_impact": {
            "cii_score": float(row["cii"]),
            "cii_tier": row["cii_tier"],
            "estimated_cost_inr_per_day": float(row["congestion_cost_rs"]),
        },
        "forecast": {
            "predicted_risk_for_tomorrow": float(row["predicted_risk"]),
            "trend": row["trend"],
        },
        "alert_level": row["alert_level"],
        "assigned_unit": row["assigned_unit"],
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "data_sent_to_scita": True,
    }

sample_payload = build_scita_payload(active_alerts.iloc[0], alert_id=1)
print("Sample SCITA payload for the top alert:\n")
print(json.dumps(sample_payload, indent=2))


Sample SCITA payload for the top alert:

{
  "alert_id": 1,
  "source_system": "AI_PARKING_INTELLIGENCE_V1",
  "h3_cell": "8861892e9bfffff",
  "junction_name": "BTP051 - Safina Plaza Junction",
  "police_station": "Shivajinagar",
  "location": {
    "lat": 12.981909468249768,
    "lon": 77.60695690792922
  },
  "violation_summary": {
    "dominant_type": "WRONG PARKING",
    "dominant_vehicle": "SCOOTER",
    "n_violations_historical": 5818,
    "total_risk_score": 29795
  },
  "congestion_impact": {
    "cii_score": 79.7,
    "cii_tier": "SEVERE",
    "estimated_cost_inr_per_day": 194561562.0
  },
  "forecast": {
    "predicted_risk_for_tomorrow": 38.2,
    "trend": "RISING"
  },
  "alert_level": "CRITICAL",
  "assigned_unit": "FKDEV00004",
  "generated_at_utc": "2026-06-20T11:00:38.950909+00:00",
  "data_sent_to_scita": true
}


In [51]:
DB_PATH = "alerts.db"

# Start fresh each run
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
conn.execute('''
CREATE TABLE alerts (
    alert_id INTEGER PRIMARY KEY AUTOINCREMENT,
    h3_cell TEXT,
    junction_name TEXT,
    police_station TEXT,
    alert_level TEXT,
    assigned_unit TEXT,
    status TEXT DEFAULT 'PENDING',
    created_at TEXT,
    scita_payload TEXT,
    scita_sync_status TEXT DEFAULT 'NOT_SENT',
    escalated INTEGER DEFAULT 0
)
''')
conn.commit()
print(f"✓ Alert database created at {DB_PATH}")


✓ Alert database created at alerts.db


In [52]:
for i, row in active_alerts.iterrows():
    payload = build_scita_payload(row, alert_id=i+1)
    conn.execute('''
        INSERT INTO alerts (h3_cell, junction_name, police_station, alert_level,
                             assigned_unit, status, created_at, scita_payload, scita_sync_status)
        VALUES (?,?,?,?,?,?,?,?,?)
    ''', (
        row["h3_cell"], row["junction_name"], row["dominant_station"], row["alert_level"],
        row["assigned_unit"], "PENDING", datetime.now(timezone.utc).isoformat(),
        json.dumps(payload), "SENT"
    ))
conn.commit()

n_inserted = conn.execute("SELECT COUNT(*) FROM alerts").fetchone()[0]
print(f"✓ {n_inserted} alerts inserted into the database and marked as sent to SCITA")


✓ 577 alerts inserted into the database and marked as sent to SCITA


In [54]:
# def format_officer_notification(row):
#     """Builds the exact push-notification text a real FCM call would send."""
#     urgency_icon = "🔴" if row["alert_level"] == "CRITICAL" else "🟡"
#     return (
#         f"{urgency_icon} [{row['alert_level']}] Patrol Alert — {row['junction_name']}\n"
#         f"   Station: {row['police_station']}  |  Unit: {row['assigned_unit']}\n"
#         f"   Predicted violations tomorrow: {row['predicted_tomorrow']:.0f} ({row['trend']})"
#     )

cur = conn.execute("SELECT * FROM alerts WHERE alert_level='CRITICAL'")
cols = [d[0] for d in cur.description]
critical_rows = pd.DataFrame(cur.fetchall(), columns=cols)

print("="*60)
print("SIMULATED FIELD NOTIFICATIONS (would be Firebase push in production)")
print("="*60)
for _, r in critical_rows.iterrows():
    print(f"\n🔴 [CRITICAL] Patrol Alert — {r['junction_name']}")
    print(f"   Station: {r['police_station']}  |  Unit: {r['assigned_unit']}")
    print(f"   Status: {r['status']}  |  SCITA sync: {r['scita_sync_status']}")


SIMULATED FIELD NOTIFICATIONS (would be Firebase push in production)

🔴 [CRITICAL] Patrol Alert — BTP051 - Safina Plaza Junction
   Station: Shivajinagar  |  Unit: FKDEV00004
   Status: PENDING  |  SCITA sync: SENT

🔴 [CRITICAL] Patrol Alert — BTP032 - Windsor Circle
   Station: Shivajinagar  |  Unit: FKDEV00012
   Status: PENDING  |  SCITA sync: SENT

🔴 [CRITICAL] Patrol Alert — BTP032 - Windsor Circle
   Station: Shivajinagar  |  Unit: FKDEV00037
   Status: PENDING  |  SCITA sync: SENT

🔴 [CRITICAL] Patrol Alert — BTP032 - Windsor Circle
   Station: Shivajinagar  |  Unit: FKDEV00041
   Status: PENDING  |  SCITA sync: SENT

🔴 [CRITICAL] Patrol Alert — BTP032 - Windsor Circle
   Station: Shivajinagar  |  Unit: FKDEV00063
   Status: PENDING  |  SCITA sync: SENT

🔴 [CRITICAL] Patrol Alert — BTP016 - 5th Main Road, RPC Layout
   Station: Vijayanagara  |  Unit: FKDEV00006
   Status: PENDING  |  SCITA sync: SENT

🔴 [CRITICAL] Patrol Alert — BTP016 - 5th Main Road, RPC Layout
   Station: Vij

In [55]:
# Simulate: backdate two alerts as if they were created 3 hours ago and never actioned
test_ids = conn.execute("SELECT alert_id FROM alerts WHERE alert_level='CRITICAL' LIMIT 2").fetchall()
backdated_time = (datetime.now(timezone.utc) - timedelta(hours=3)).isoformat()
for (alert_id,) in test_ids:
    conn.execute("UPDATE alerts SET created_at=? WHERE alert_id=?", (backdated_time, alert_id))
conn.commit()
print(f"✓ Simulated {len(test_ids)} CRITICAL alerts sitting unaddressed for 3 hours")


✓ Simulated 2 CRITICAL alerts sitting unaddressed for 3 hours


In [56]:
ESCALATION_THRESHOLD_HOURS = 2  # CRITICAL alerts unactioned beyond this get escalated

cur = conn.execute("SELECT alert_id, junction_name, created_at, status FROM alerts WHERE alert_level='CRITICAL' AND status='PENDING'")
candidates = cur.fetchall()

escalated_count = 0
for alert_id, junction, created_at, status in candidates:
    age_hours = (datetime.now(timezone.utc) - datetime.fromisoformat(created_at)).total_seconds() / 3600
    if age_hours >= ESCALATION_THRESHOLD_HOURS:
        conn.execute("UPDATE alerts SET status='ESCALATED', escalated=1 WHERE alert_id=?", (alert_id,))
        escalated_count += 1
        print(f"⬆ ESCALATED — Alert #{alert_id} ({junction}) — pending for {age_hours:.1f}h, exceeds {ESCALATION_THRESHOLD_HOURS}h threshold")

conn.commit()
print(f"\n✓ {escalated_count} alert(s) escalated to station duty officer")


⬆ ESCALATED — Alert #1 (BTP051 - Safina Plaza Junction) — pending for 3.0h, exceeds 2h threshold
⬆ ESCALATED — Alert #2 (BTP032 - Windsor Circle) — pending for 3.0h, exceeds 2h threshold

✓ 2 alert(s) escalated to station duty officer


In [57]:
# Verify the escalation actually persisted in the database
final_state = pd.read_sql("SELECT alert_id, junction_name, alert_level, status, escalated FROM alerts", conn)
print("Final alert status distribution:")
print(final_state["status"].value_counts())
print()
final_state[final_state["escalated"]==1]


Final alert status distribution:
status
PENDING      575
ESCALATED      2
Name: count, dtype: int64



,alert_id,junction_name,alert_level,status,escalated
0,1,BTP051 - Safina Plaza Junction,CRITICAL,ESCALATED,1
1,2,BTP032 - Windsor Circle,CRITICAL,ESCALATED,1


In [58]:
OUTPUT_DIR = Path(".")

full_log = pd.read_sql("SELECT * FROM alerts", conn)
full_log.to_csv(OUTPUT_DIR / "phase5_alert_log.csv", index=False)
conn.close()

print(f"✓ Alert log exported: {OUTPUT_DIR / 'phase5_alert_log.csv'}  ({len(full_log)} alerts)")
print(f"✓ Alert database file remains at: {DB_PATH}")


✓ Alert log exported: phase5_alert_log.csv  (577 alerts)
✓ Alert database file remains at: alerts.db
